In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_PATH = "../data/processed/menganti_CO_daily.csv"
OUTPUT_PATH = "../data/processed/menganti_CO_preprocessed.csv"

df = pd.read_csv(INPUT_PATH)

df["date"] = pd.to_datetime(df["date"])
df["CO"] = pd.to_numeric(df["CO"], errors="coerce")

df = df.sort_values("date").reset_index(drop=True)

print("Jumlah baris :", len(df))
print("Missing CO   :", df["CO"].isna().sum())
print("Tanggal awal :", df["date"].min())
print("Tanggal akhir:", df["date"].max())

Jumlah baris : 366
Missing CO   : 110
Tanggal awal : 2025-08-31 00:00:00
Tanggal akhir: 2026-08-31 00:00:00


In [6]:
Q1 = df["CO"].quantile(0.25)
Q3 = df["CO"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (df["CO"] < lower_bound) |
    (df["CO"] > upper_bound)
)

print("Q1              :", Q1)
print("Q3              :", Q3)
print("IQR             :", IQR)
print("Batas bawah     :", lower_bound)
print("Batas atas      :", upper_bound)
print("Jumlah outlier  :", outlier_mask.sum())

print("\nData outlier:")
display(df.loc[outlier_mask, ["date", "CO"]])

Q1              : 0.027062103644545574
Q3              : 0.031294239086231974
IQR             : 0.004232135441686401
Batas bawah     : 0.020713900482015973
Batas atas      : 0.03764244224876158
Jumlah outlier  : 7

Data outlier:


,date,CO
24,2025-09-24,0.041763
37,2025-10-07,0.043049
39,2025-10-09,0.038615
40,2025-10-10,0.038976
326,2026-07-23,0.038878
348,2026-08-14,0.037996
362,2026-08-28,0.037915


In [7]:
df.loc[outlier_mask, "CO"] = np.nan

print("Missing sebelum preprocessing :", 110)
print("Missing setelah outlier       :", df["CO"].isna().sum())

Missing sebelum preprocessing : 110
Missing setelah outlier       : 117


In [8]:
df = df.set_index("date")

df["CO"] = (
    df["CO"]
    .interpolate(method="time")
    .ffill()
    .bfill()
)

df = df.reset_index()

print("Missing setelah imputasi :", df["CO"].isna().sum())
print("Jumlah baris             :", len(df))
print("Tanggal awal             :", df["date"].min())
print("Tanggal akhir            :", df["date"].max())

Missing setelah imputasi : 0
Jumlah baris             : 366
Tanggal awal             : 2025-08-31 00:00:00
Tanggal akhir            : 2026-08-31 00:00:00


In [9]:
df.to_csv(OUTPUT_PATH, index=False)

print(f"File tersimpan: {OUTPUT_PATH}")

File tersimpan: ../data/processed/menganti_CO_preprocessed.csv
